# Mine small RDF snapshots

Retrieve bounded two-hop data through SparqlHelper, then run the package miner locally. Keep source queries, RDF, and canonical MinedSchema JSON together. These are development samples.

In [ ]:
from pathlib import Path
import json, os
from rdfsolve import SchemaMiner
from rdfsolve.sparql_helper import SparqlHelper

folder = Path(os.environ["RDFSOLVE_ROOT"]) / "notebooks/mcp/schemas"
folder.mkdir(exist_ok=True)
sources = [
    ("aopwikirdf-small", "https://aopwiki.rdf.bigcat-bioinformatics.org/sparql", "http://aopkb.org/aop_ontology#AdverseOutcomePathway"),
    ("wikipathways-small", "https://sparql.wikipathways.org/sparql", "http://vocabularies.wikipathways.org/wp#Pathway"),
]

In [ ]:
for name, endpoint, root_type in sources:
    query = f"""CONSTRUCT {{ ?s ?p ?o . ?o ?q ?v . ?v a ?type . }} WHERE {{
      {{ SELECT ?s WHERE {{ ?s a <{root_type}> }} ORDER BY ?s LIMIT 8 }}
      ?s ?p ?o .
      OPTIONAL {{ FILTER(isIRI(?o)) ?o ?q ?v . OPTIONAL {{ ?v a ?type }} }}
    }}"""
    with SparqlHelper(endpoint, timeout=120, inter_request_delay=0.5) as helper:
        graph = helper.construct_graph(query)
    graph.serialize(destination=folder / f"{name}.ttl", format="turtle")
    with SchemaMiner.from_graph(graph, endpoint_url=endpoint, counts=False,
                                strategy="one-shot", delay=0, enrich=True) as miner:
        schema = miner.mine(name)
    schema.about.description = "Bounded two-hop snapshot rooted at eight resources; evaluate with the retained local RDF file."
    (folder / f"{name}.schema.json").write_text(json.dumps(schema.to_dict(), indent=2))
    (folder / f"{name}.source.rq").write_text(query)
    print(name, len(graph), "triples;", len(schema.patterns), "patterns")

In [ ]:
from rdflib import Graph
local = Graph().parse(data="""@prefix e: <urn:study:> .
e:alice a e:Researcher; e:employment e:j1, e:j2 . e:bob a e:Researcher; e:employment e:j3 .
e:j1 a e:Employment; e:organisation e:X; e:year 2010 .
e:j2 a e:Employment; e:organisation e:Y; e:year 2020 .
e:j3 a e:Employment; e:organisation e:X; e:year 2020 .
e:X a e:Organisation . e:Y a e:Organisation .""", format='turtle')
local.serialize(destination=folder / 'local-study.ttl', format='turtle')
with SchemaMiner.from_graph(local, counts=False, strategy='one-shot', delay=0) as miner:
    schema = miner.mine('local-study')
(folder / 'local-study.schema.json').write_text(json.dumps(schema.to_dict(), indent=2))
print('local-study', len(local), 'triples;', len(schema.patterns), 'patterns')